# Desafio técnico Bradesco DE 

**Autor:** Felipe Piva

**Dataset:** Activity recognition exp.zip (4 CSVs: Phones/Watch x accelerometer/gyroscope)

## Estrutura do notebook (ETL)

0. Setup          

1. RAW - Ingestão 

2. Problemas encontrados e possíveis soluções

3. Solução dos problemas encontrados 

4. Sumarização por usuário

5. Desafios opcionais

## Seção 0 - Setup 

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.window import Window

# Caminho para o dataset
RAW_PATH = '/Volumes/bradesco_desafio/desafio/raw/'

## Seção 1 - ingestão 


In [0]:
# Schema explícito (sem inferSchema)
schema = StructType([
    StructField("Index",         LongType(),   True),
    StructField("Arrival_Time",  LongType(),   True),
    StructField("Creation_Time", LongType(),   True),
    StructField("x",             DoubleType(), True),
    StructField("y",             DoubleType(), True),
    StructField("z",             DoubleType(), True),
    StructField("User",          StringType(), True),
    StructField("Model",         StringType(), True),
    StructField("Device",        StringType(), True),
    StructField("gt",            StringType(), True),
])

# mapa de arquivos (sensor + tipo de device)
files = {
    "Phones_accelerometer.csv" : ("phone", "accelerometer"),
    "Phones_gyroscope.csv"     : ("phone", "gyroscope"),
    "Watch_accelerometer.csv"  : ("watch", "accelerometer"),
    "Watch_gyroscope.csv"      : ("watch", "gyroscope")
}

def read_csv(
    filename: str,
    device_type: str,
    sensor : str
):
    return (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(f"{RAW_PATH}{filename}")
        .withColumn("device_type", F.lit(device_type))
        .withColumn("sensor",      F.lit(sensor))
        .withColumn("source_file", F.lit(filename))
    )

df_raw = None

for filename, (device_type, sensor) in files.items():
    df = read_csv(filename, device_type, sensor)
    df_raw = df if df_raw is None else df_raw.unionByName(df)

print(f"Total de registros lidos: {df_raw.count()}")

## Seção 2 - Análise inicial / Problemas encontrados

In [0]:
display(df_raw)

In [0]:
# Problema 1 - Coluna target 'gt' com nulos e a string literal "null"
(
    df_raw
    .groupBy("gt")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

print(f"Total de registros reais null: {df_raw.filter(F.col("gt").isNull()).count()}")
print(f"Total de registro com string null: {df_raw.filter(F.col("gt") == "null").count()}")

In [0]:
# Problema 2 - Inconsitência na Creation Date enunciado afirma que os dados são gravados em nanosegundos mas na pratica alguns devices gravam em milisegundos
# Comparar com o Arrival Time revela isso
(
    df_raw
    .groupBy("Device")
    .agg(
        F.min("Creation_Time").alias("min_creation"),
        F.max("Creation_Time").alias("max_creation"),
        F.min("Arrival_Time").alias("min_arrival"),
        F.max("Arrival_Time").alias("max_arrival"),
    )
    .show(50, truncate=False)
)

In [0]:
# Problema 3 - Dados com mesmo timestamp e mesmo user e device - indicando dados dupllicados 
dups = (
    df_raw
    .groupBy("User", "Device", "Creation_Time", "source_file")
    .count()
    .filter(F.col("count") > 1)
    .select("User", "Device", "Creation_Time", "source_file")
)

df_dups = (
    df_raw
    .join(dups, on=["User", "Device", "Creation_Time", "source_file"], how="inner")
)

display(df_dups)

In [0]:
# Problema 4 - Qualidade do campo Creation_Time (investigação).
# A suspeita inicial era "dessincronia de relógio" (Arrival < Creation). Porém o
# Creation_Time cru varia em várias ordens de grandeza (faixas de 12 a 19 dígitos),
# inclusive dentro do mesmo Device.
(
    df_raw
    .withColumn("dig_creation", F.length(F.col("Creation_Time").cast("string")))
    .groupBy("dig_creation")
    .count()
    .orderBy("dig_creation")
    .show(50, truncate=False)
)

### Resumo dos problemas e como tratar:

- **Problema 1:** Coluna target (`gt`) com valores faltantes - a string literal "null"
  convive com nulos reais (registros de transição entre atividades).
  Solução: normalizar "null" → NULL; manter ou filtrar conforme o uso final.

- **Problema 2:** `Creation_Time` é um campo **não-confiável** para uso temporal. A magnitude
  varia em 5 faixas de dígitos (12, 13, 14, 15 e 19), inclusive dentro do mesmo `Device`, e
  não corresponde a unidades canônicas separadas por potências de 10 limpas. Uma tentativa de
  normalização por dígitos foi testada e **rejeitada com evidência**: alinhou a escala mas não
  o valor (81% dos registros permaneceram com `Arrival < Creation` após normalizar).
  Solução: usar `Arrival_Time` (consistente em 13 dígitos = ms) como **única** referência
  temporal; manter `Creation_Time` cru apenas para auditoria.

- **Problema 3:** Duplicatas na chave de identificação dos registros.
  Solução: de-duplicar (remover) os registros repetidos, garantindo consistência na base final.

- **Problema 4:** A "suspeita de dessincronia de relógio" (Arrival < Creation) não
  se sustenta como diagnóstico, porque dependeria de comparar contra um campo comprovadamente
  podre (Problema 2). O problema real e comprovado é a **qualidade do `Creation_Time`**: sua
  magnitude é inconsistente entre e dentro de devices. Solução: isolar `Arrival_Time` como
  referência temporal e não derivar conclusões temporais de `Creation_Time`.

## Seção 3 - Solução das inconsitências encontradas no dataset

In [0]:
df_stg = df_raw

# Solução 1 - normalizar a string literal "null" para NULL real
df_stg = df_stg.withColumn(
    "gt",
    F.when(F.col("gt") == "null", F.lit(None)).otherwise(F.col("gt"))
)

# Solução 2 - Creation_Time: campo NÃO CONFIÁVEL para uso temporal.
# Investigação (ver Seção 2): a magnitude varia em 5 faixas de dígitos (12,13,14,15,19),
# inclusive dentro do mesmo device, e não corresponde a unidades comuns separadas por
# potências de 10.

# Solução 3 - remover duplicados pela chave de negócio (timestamp CRU, não normalizado)
df_stg = df_stg.dropDuplicates(["User", "Device", "Creation_Time", "source_file"])

# Solução 4 - REMOVIDA. A flag original (Arrival < Creation) diagnosticava dessincronia de
# relógio em cima de um campo comprovadamente não-confiável (Solução 2) - seria construir uma
# conclusão sobre dado podre.

print(f"Registros após de-duplicação: {df_stg.count()}")

In [0]:
# Conferência: nº de dígitos por Device.
# Arrival_Time é estável (13 dígitos = ms) -> confiável.
# Creation_Time cru oscila de magnitude -> evidência da decisão da Solução 2.
(df_stg
 .groupBy("Device")
 .agg(
     F.min(F.length(F.col("Creation_Time").cast("string"))).alias("dig_creation_min"),
     F.max(F.length(F.col("Creation_Time").cast("string"))).alias("dig_creation_max"),
     F.min(F.length(F.col("Arrival_Time").cast("string"))).alias("dig_arrival_min"),
     F.max(F.length(F.col("Arrival_Time").cast("string"))).alias("dig_arrival_max"),
 )
 .orderBy("Device")
 .show(50, False))

## Seção 4 — Sumarização por usuário

In [0]:
df_user_summary = (
    df_stg.groupBy("User").agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("Model").alias("qtd_modelos"),
        F.collect_set("Model").alias("modelos"),
        F.countDistinct("Device").alias("qtd_devices"),
        F.collect_set("Device").alias("devices"),
        F.countDistinct("gt").alias("qtd_atividades"),
        F.collect_set("gt").alias("atividades"),
        F.sum(F.when(F.col("sensor") == "accelerometer", 1).otherwise(0)).alias("reg_acelerometro"),
        F.sum(F.when(F.col("sensor") == "gyroscope", 1).otherwise(0)).alias("reg_giroscopio"),
    )
    .orderBy("User")
)
 
display(df_user_summary)

## Seção 5 - Desafios opcionais

In [0]:
# Opcional 1 - Timestamp legível.
# Convertemos apenas Arrival_Time (ms -> datetime). Creation_Time não é convertido por falta de confiabilidade.
df_stg = df_stg.withColumn(
    "Arrival_DateTime", F.to_timestamp(F.col("Arrival_Time") / 1000)
)

df_stg.select("Arrival_Time", "Arrival_DateTime").show(10, truncate=False)

In [0]:
# Opcional 2 - Diferença de tempo entre registros consecutivos.
# Ordenamos por Arrival_Time porque é a referência temporal confiável que a análise identificou
# (Creation_Time foi descartado como base temporal na Solução 2).
w = Window.partitionBy("User", "Device", "source_file").orderBy(
    "Arrival_Time", "source_file", "Index"
)

df_stg = (
    df_stg
    .withColumn("prev_Arrival_Time", F.lag("Arrival_Time").over(w))
    .withColumn("delta_ms", F.col("Arrival_Time") - F.col("prev_Arrival_Time"))
    .withColumn("delta_seg", F.col("delta_ms") / 1000)
)

# Conferência: deltas por User+Device
df_stg.select(
    "User", "Device", "Arrival_DateTime",
    "source_file", "Index",
    "delta_ms", "delta_seg"
).orderBy("User", "Device", "Arrival_Time", "source_file", "Index").show(20, truncate=False)

In [0]:
# Diagnóstico de qualidade do Creation_Time (evidência da Solução 2).
# Em vez de "flagar dessincronia" (que pressuporia Creation confiável), medimos ONDE a
# inconsistência Arrival < Creation se concentra, por FAIXA DE DÍGITOS do Creation cru.
# Resultado esperado: a inconsistência se espalha por múltiplas faixas -> normalizar por
# potência de 10 não resolve -> campo não-confiável, conforme decidido.
(df_stg
 .withColumn("dig_creation", F.length(F.col("Creation_Time").cast("string")))
 .withColumn(
     "arrival_menor_que_creation",
     F.when(F.col("Arrival_Time") < F.col("Creation_Time"), F.lit(1)).otherwise(F.lit(0))
 )
 .groupBy("dig_creation")
 .agg(
     F.count("*").alias("total"),
     F.sum("arrival_menor_que_creation").alias("inconsistentes"),
 )
 .withColumn("pct_inconsistente", F.round(F.col("inconsistentes") / F.col("total") * 100, 2))
 .orderBy("dig_creation")
 .show(50, False))